# Simulación Monte Carlo: análisis de dos diseños de bombas de pistón

Este notebook resuelve los dos ejercicios del documento Montecarlo 1 y 2 en un mismo flujo. Cada caso conserva su planteamiento, define las variables, simula 100,000 observaciones, visualiza la distribución del caudal y responde si conviene cambiar el diseño o si el proyecto está en riesgo.

Se utiliza Python con NumPy, pandas, Matplotlib, Seaborn y SciPy.

## Técnica y método

La simulación Monte Carlo representa la variación natural de un proceso mediante números aleatorios. Para cada repetición se toma un valor posible de cada variable de entrada y se calcula el resultado del sistema.

En estos ejercicios:

1. Se modelan las dimensiones y RPM con distribuciones normales.
2. Se calcula el caudal de cada bomba.
3. Se repite el cálculo 100,000 veces.
4. Se calcula media, desviación estándar, percentiles y proporción dentro de especificación.
5. Se interpreta el riesgo contra el objetivo del diseño.

La media permite saber si el caudal está centrado en el objetivo. La desviación estándar permite saber qué tan variable es. Los percentiles y la proporción dentro de especificación muestran el riesgo de que una bomba quede fuera del rango deseado.

## Importante sobre las unidades

En el ejercicio 1, las dimensiones están en milímetros. El volumen geométrico se convierte a mililitros dividiendo entre 1,000, porque 1 ml = 1 cm cúbico y 1,000 mm cúbicos = 1 ml.

En el ejercicio 2, las dimensiones están en centímetros. Como 1 cm cúbico equivale a 1 ml, el resultado de área por carrera ya queda en ml por carrera.

RPM significa revoluciones por minuto. Al multiplicar ml por carrera por RPM se obtiene ml/min.

La desviación estándar de caudal se expresa en ml/min, aunque el PDF escribe “0.2 ml”. En el notebook se interpreta como 0.2 ml/min porque el caudal es una tasa.

## 1. Configuración común

Se importan las librerías y se fija una semilla. La semilla no cambia la lógica del ejercicio; sólo permite reproducir los mismos 100,000 escenarios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
N_SIMULACIONES = 100_000
sns.set_theme(style='whitegrid')
print('Simulaciones por ejercicio:', N_SIMULACIONES)
print('Semilla:', RANDOM_STATE)

### Explicación del código

NumPy genera muestras aleatorias y realiza los cálculos vectorizados. pandas organiza las simulaciones en tablas. Matplotlib y Seaborn construyen las gráficas. SciPy permite comparar la distribución simulada con una distribución normal teórica.

Trabajar con arreglos completos permite simular 100,000 datos rápidamente y evita escribir un ciclo individual para cada bomba.

# Ejercicio 1. Bomba con radio de pistón y retorno de flujo

## Descripción original del ejercicio

**Problema. 1. Simulación Montecarlo.**

Una empresa de manufactura que se dedica al desarrollo de nuevos productos necesita evaluar el diseño de un producto propuesto: El producto consta de una bomba de pistón pequeña que debe bombear 10ml de fluido por minuto. Usted desea estimar el Rendimiento probable de miles de bombas, dados los siguientes parámetros:

Caudal (en ml) = [π * R² * L / 1000 – B] * RPM

Para facilitar el ejercicio supongamos lo siguiente: Basándose en el rendimiento de otras bombas que ha manufacturado su empresa, Usted puede decir que el radio (R) del pistón se distribuye normalmente con una media de 2.5 mm y una desviación estándar de 0.009 mm.

La longitud de la carrera (L) se distribuye normalmente con una media de 12.6 mm y una desviación estándar de 0.22 mm.

El retorno de flujo (B) se distribuye normalmente con una media de 0.05 y una desviación estándar de 0.0025.

Por último, las carreras por minuto (RPM) se distribuyen normalmente con una media de 50.1 RPM y una desviación estándar de 1.1 RPM. Simular 100,000 datos.

Tu objetivo es cumplir con un flujo de 10 ml/min ± 1ml/min.

**¿Habrá que cambiar el diseño? ¿El proyecto está en riesgo?**

**Justifica tu respuesta.**

## 1.1 Interpretación del modelo físico y de sus unidades

R es el radio en mm. L es la carrera en mm. El término πR²L representa el volumen geométrico de una carrera en mm cúbicos. La división entre 1,000 lo convierte a ml por carrera.

B representa el retorno de flujo y se interpreta como ml por carrera. Se resta porque disminuye el volumen neto bombeado.

RPM representa carreras o revoluciones por minuto. Por eso el volumen neto por carrera se multiplica por RPM y se obtiene el caudal en ml/min.

El rango de aceptación es de 9 a 11 ml/min. Un escenario dentro de ese intervalo cumple el objetivo.

## 1.2 Simulación del ejercicio 1

Se generan 100,000 valores para radio, carrera, retorno y RPM. Cada posición de los arreglos representa una bomba simulada.

In [ ]:
R1 = rng.normal(2.5, 0.009, N_SIMULACIONES)
L1 = rng.normal(12.6, 0.22, N_SIMULACIONES)
B1 = rng.normal(0.05, 0.0025, N_SIMULACIONES)
RPM1 = rng.normal(50.1, 1.1, N_SIMULACIONES)

caudal1 = (np.pi * R1**2 * L1 / 1000 - B1) * RPM1
limite_inferior1, limite_superior1 = 9, 11
cumple1 = (caudal1 >= limite_inferior1) & (caudal1 <= limite_superior1)

ejercicio1 = pd.DataFrame({
    'radio_mm': R1,
    'carrera_mm': L1,
    'retorno_ml_carrera': B1,
    'rpm': RPM1,
    'caudal_ml_min': caudal1,
    'cumple_especificacion': cumple1
})
print('Simulaciones:', len(ejercicio1))
display(ejercicio1.head())

### Explicación detallada del código

Cada llamada a rng.normal genera 100,000 valores con la media y desviación estándar del PDF. El cálculo se aplica elemento por elemento: el primer radio se combina con la primera carrera, el primer retorno y las primeras RPM.

La fórmula conserva la división entre 1,000 para convertir mm cúbicos a ml. La variable cumple_especificacion es True sólo cuando el caudal queda entre 9 y 11 ml/min.

## 1.3 Validación de entradas y unidades

Antes de interpretar el caudal comprobamos que las entradas sean físicamente razonables y que la fórmula produzca valores positivos.

In [ ]:
print({
    'radios_no_positivos': int((R1 <= 0).sum()),
    'carreras_no_positivas': int((L1 <= 0).sum()),
    'retornos_no_positivos': int((B1 <= 0).sum()),
    'rpm_no_positivas': int((RPM1 <= 0).sum()),
    'caudales_no_positivos': int((caudal1 <= 0).sum())
})
display(ejercicio1.describe().round(4))

### Interpretación

Los conteos de valores no positivos deben ser cero o prácticamente cero. Las medias y desviaciones deben ser cercanas a los parámetros proporcionados en el documento.

El retorno B debe ser menor que el volumen geométrico por carrera para que exista caudal neto positivo. Esta comprobación también ayuda a descubrir errores de conversión.

## 1.4 Distribución del caudal del ejercicio 1

Esta gráfica muestra el rendimiento probable de las bombas y el intervalo de aceptación.

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(caudal1, bins=80, kde=True, color='#4C78A8', stat='density')
plt.axvspan(9, 11, color='#2CA02C', alpha=.18, label='Especificación: 9 a 11 ml/min')
plt.axvline(caudal1.mean(), color='#D62728', linestyle='--', label='Media simulada')
plt.title('Ejercicio 1: distribución simulada del caudal')
plt.xlabel('Caudal (ml/min)')
plt.ylabel('Densidad')
plt.legend()
plt.show()

### Interpretación visual

La zona verde representa bombas que cumplen. La línea roja representa la media. Si la distribución queda centrada dentro de la zona verde y la mayor parte de su área está allí, el diseño tiene buen rendimiento probable.

Las colas fuera de la zona verde representan bombas que podrían entregar demasiado o muy poco fluido.

## 1.5 Resultados estadísticos del ejercicio 1

In [ ]:
media1 = caudal1.mean()
std1 = caudal1.std(ddof=1)
p01, p05, p50, p95, p99 = np.percentile(caudal1, [1,5,50,95,99])
porcentaje_cumple1 = cumple1.mean()
print(f'Media: {media1:.4f} ml/min')
print(f'Desviación estándar: {std1:.4f} ml/min')
print(f'Percentil 1: {p01:.4f} ml/min')
print(f'Percentil 5: {p05:.4f} ml/min')
print(f'Mediana: {p50:.4f} ml/min')
print(f'Percentil 95: {p95:.4f} ml/min')
print(f'Percentil 99: {p99:.4f} ml/min')
print(f'Porcentaje dentro de especificación: {porcentaje_cumple1:.2%}')
print(f'Porcentaje fuera de especificación: {(1-porcentaje_cumple1):.2%}')

### Interpretación detallada del ejercicio 1

La media se compara con 10 ml/min. La desviación estándar muestra la variación entre bombas. El porcentaje dentro de especificación responde directamente cuántas bombas simuladas cumplen el intervalo de 9 a 11 ml/min.

Si el porcentaje de cumplimiento es alto, el proyecto tiene bajo riesgo desde este modelo. Si la media está desplazada o las colas salen ampliamente del intervalo, se debe cambiar el diseño o reducir la variación de sus componentes.

No se debe concluir sólo con la media: una media correcta puede ocultar demasiadas bombas fuera de especificación.

## 1.6 Sensibilidad del caudal a las entradas

Los gráficos de dispersión muestran qué variables se relacionan más con el caudal. Esto ayuda a decidir dónde enfocar tolerancias y control de proceso.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14,10))
for ax, x, nombre in [
    (axes[0,0], R1, 'Radio (mm)'),
    (axes[0,1], L1, 'Carrera (mm)'),
    (axes[1,0], B1, 'Retorno (ml/carrera)'),
    (axes[1,1], RPM1, 'RPM')
]:
    ax.scatter(x, caudal1, s=3, alpha=.15, color='#4C78A8')
    ax.set_xlabel(nombre); ax.set_ylabel('Caudal (ml/min)')
    ax.set_title('Caudal vs ' + nombre)
plt.tight_layout()
plt.show()

### Interpretación

El radio tiene un efecto fuerte porque aparece elevado al cuadrado. La carrera y RPM tienen efectos aproximadamente proporcionales. El retorno reduce el caudal.

La variable con mayor relación visual y mayor impacto físico es candidata a recibir la tolerancia más estricta o un mejor sistema de medición.

# Ejercicio 2. Bomba con diámetro de pistón

## Descripción original del ejercicio

**Problema. 2. Simulación Montecarlo.**

Una empresa de manufactura que se dedica al desarrollo de nuevos productos necesita evaluar el diseño de un producto propuesto: El producto consta de una bomba de pistón pequeña que debe bombear 12ml de fluido por minuto. Usted desea estimar el Rendimiento probable de miles de bombas, dados los siguientes parámetros:

La variación natural del diámetro del pistón (D), longitud de la carrera (L) y carreras por minuto (RPM). Lo ideal es que la desviación estándar del caudal de la bomba en miles de bombas no sea mayor de 0.2 ml.

Caudal (en ml) = π (D/2)² * L * RPM

Para facilitar el ejercicio supongamos lo siguiente: Basándose en el rendimiento de otras bombas que ha manufacturado su empresa, Usted puede decir que el diámetro del pistón se distribuye normalmente con una media de 0.8 cm y una desviación estándar de 0.0003 cm. La longitud de la carrera se distribuye normalmente con una media de 2.5 cm y una desviación estándar de 0.15 cm. Por último, las carreras por minuto se distribuyen normalmente con una media de 9.549 RPM y una desviación estándar de 0.17 RPM. Simular 100,000 datos.

Lo ideal es que la desviación estándar del caudal de la bomba en miles de bombas no sea mayor de 0.2ml.

**¿Habrá que cambiar el diseño? ¿El proyecto está en riesgo?**

**Justifica tu respuesta.**

## 2.1 Interpretación del modelo físico y de sus unidades

D está en centímetros y representa el diámetro, no el radio. Por eso se divide entre 2 para obtener el radio. L también está en centímetros.

El área del pistón es π(D/2)² y queda en cm². Al multiplicar por L se obtiene cm³ por carrera. Como 1 cm³ = 1 ml, no se necesita dividir entre 1,000.

Al multiplicar por RPM, el caudal queda en ml/min. El criterio principal del ejercicio 2 es que la desviación estándar del caudal no sea mayor que 0.2 ml/min. También se revisa que la media esté alrededor del objetivo de 12 ml/min.

## 2.2 Simulación del ejercicio 2

In [ ]:
D2 = rng.normal(0.8, 0.0003, N_SIMULACIONES)
L2 = rng.normal(2.5, 0.15, N_SIMULACIONES)
RPM2 = rng.normal(9.549, 0.17, N_SIMULACIONES)

caudal2 = np.pi * (D2 / 2)**2 * L2 * RPM2
objetivo2 = 12
std_maxima2 = 0.2

ejercicio2 = pd.DataFrame({
    'diametro_cm': D2,
    'carrera_cm': L2,
    'rpm': RPM2,
    'caudal_ml_min': caudal2
})
print('Simulaciones:', len(ejercicio2))
display(ejercicio2.head())

### Explicación detallada del código

Se generan 100,000 diámetros, carreras y RPM con las medias y desviaciones del PDF. El diámetro se divide entre 2 antes de elevarlo al cuadrado porque la fórmula requiere radio.

No se divide entre 1,000: las dimensiones están en centímetros y cm³ equivale a ml. Esta diferencia de unidades es la principal distinción entre las fórmulas de los dos ejercicios.

## 2.3 Validación de entradas del ejercicio 2

In [ ]:
print({
    'diametros_no_positivos': int((D2 <= 0).sum()),
    'carreras_no_positivas': int((L2 <= 0).sum()),
    'rpm_no_positivas': int((RPM2 <= 0).sum()),
    'caudales_no_positivos': int((caudal2 <= 0).sum())
})
display(ejercicio2.describe().round(4))

### Interpretación

Las entradas deben ser positivas. La media del caudal debe estar cerca de 12 ml/min. La desviación estándar será comparada con el máximo permitido de 0.2 ml/min.

## 2.4 Distribución del caudal del ejercicio 2

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(caudal2, bins=80, kde=True, color='#9467BD', stat='density')
plt.axvline(objetivo2, color='#2CA02C', linestyle='--', linewidth=2, label='Objetivo: 12 ml/min')
plt.axvline(caudal2.mean(), color='#D62728', linestyle='--', label='Media simulada')
plt.title('Ejercicio 2: distribución simulada del caudal')
plt.xlabel('Caudal (ml/min)')
plt.ylabel('Densidad')
plt.legend()
plt.show()

### Interpretación visual

La línea verde muestra el objetivo. La línea roja muestra el centro realista de la producción simulada. La anchura de la curva representa la variación natural entre bombas.

El criterio no es sólo que la curva esté centrada en 12 ml/min: también debe ser suficientemente estrecha para que su desviación estándar sea como máximo 0.2 ml/min.

## 2.5 Resultados estadísticos del ejercicio 2

In [ ]:
media2 = caudal2.mean()
std2 = caudal2.std(ddof=1)
percentiles2 = np.percentile(caudal2, [1,5,50,95,99])
print(f'Media: {media2:.4f} ml/min')
print(f'Desviación estándar: {std2:.4f} ml/min')
print('Percentiles 1, 5, 50, 95 y 99:', np.round(percentiles2, 4))
print(f'Cumple desviación estándar <= 0.2 ml/min: {std2 <= std_maxima2}')
print(f'Diferencia de la media contra 12 ml/min: {media2-objetivo2:.4f} ml/min')

### Interpretación detallada del ejercicio 2

La media indica si el diseño está centrado en el caudal objetivo. La desviación estándar es el criterio explícito del PDF: debe ser menor o igual a 0.2 ml/min.

Si la desviación es menor o igual a 0.2, el diseño cumple el requisito de variabilidad. Si además la media está cerca de 12 ml/min, el proyecto presenta bajo riesgo bajo estos supuestos. Si la desviación excede 0.2, habría que reducir la variación del diámetro, carrera o RPM.

## 2.6 Sensibilidad del ejercicio 2

Analizamos visualmente el efecto del diámetro, la carrera y RPM sobre el caudal. El diámetro puede tener un efecto importante porque el área depende del diámetro al cuadrado.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17,5))
for ax, x, nombre in [
    (axes[0], D2, 'Diámetro (cm)'),
    (axes[1], L2, 'Carrera (cm)'),
    (axes[2], RPM2, 'RPM')
]:
    ax.scatter(x, caudal2, s=3, alpha=.15, color='#9467BD')
    ax.set_xlabel(nombre); ax.set_ylabel('Caudal (ml/min)')
    ax.set_title('Caudal vs ' + nombre)
plt.tight_layout()
plt.show()

### Interpretación

La pendiente y la dispersión de cada gráfico muestran qué entrada puede explicar más variación. Aunque la desviación del diámetro sea pequeña, su efecto puede ser relevante porque aparece al cuadrado. La carrera y RPM afectan proporcionalmente.

Esta información orienta una decisión de ingeniería: controlar la variable más influyente puede reducir la desviación del caudal con menor esfuerzo que controlar todas por igual.

## 3. Comparación y conclusiones de ambos ejercicios

Se resumen los indicadores principales para apoyar la decisión.

In [ ]:
comparacion = pd.DataFrame({
    'ejercicio': ['Ejercicio 1', 'Ejercicio 2'],
    'media_caudal_ml_min': [media1, media2],
    'desviacion_estandar_ml_min': [std1, std2],
    'objetivo_ml_min': [10, 12],
    'criterio_principal': ['Entre 9 y 11 ml/min', 'Desviación <= 0.2 ml/min'],
    'resultado': [
        'Cumplimiento del intervalo: ' + f'{porcentaje_cumple1:.1%}',
        'Desviación cumple: ' + str(std2 <= std_maxima2)
    ]
})
display(comparacion.round(4))

### Interpretación final

El ejercicio 1 se decide principalmente por el porcentaje de bombas dentro del intervalo de 9 a 11 ml/min. La media y la desviación ayudan a explicar por qué algunas bombas quedan fuera.

El ejercicio 2 se decide principalmente por la desviación estándar del caudal. Una media cercana a 12 ml/min no basta si la dispersión supera 0.2 ml/min.

En ambos casos, el riesgo calculado depende de que las medias, desviaciones y distribuciones del PDF representen correctamente el proceso real.

## 4. Conclusiones generales

- Monte Carlo permite estimar el rendimiento de miles de bombas sin fabricar las 100,000 unidades.
- El ejercicio 1 requiere un caudal entre 9 y 11 ml/min.
- El ejercicio 2 requiere controlar la desviación estándar del caudal a un máximo de 0.2 ml/min.
- Las conversiones de unidades son esenciales: el ejercicio 1 usa mm y divide entre 1,000; el ejercicio 2 usa cm y no necesita esa conversión.
- El diámetro o radio merece atención especial porque el área depende del cuadrado de la dimensión.
- La decisión de cambiar el diseño debe considerar el porcentaje fuera de especificación, la desviación, el costo de reducir variación y la capacidad de control del proceso.
- Antes de aprobar un diseño real se deben validar los supuestos con datos de prototipos y analizar posibles correlaciones entre variables.

## 5. Resumen reproducible

In [ ]:
print({
    'ejercicio_1_media_ml_min': round(float(media1),4),
    'ejercicio_1_std_ml_min': round(float(std1),4),
    'ejercicio_1_porcentaje_cumple': round(float(porcentaje_cumple1),4),
    'ejercicio_2_media_ml_min': round(float(media2),4),
    'ejercicio_2_std_ml_min': round(float(std2),4),
    'ejercicio_2_cumple_std_max_0_2': bool(std2 <= std_maxima2)
})